# Banking Intent — Fine-tune Qwen3-8B on Google Colab

**Before running:**
1. Runtime → Change runtime type → **A100** (or L4)
2. Colab → left sidebar → 🔑 Secrets: add `HF_TOKEN` (HuggingFace token with write access)
3. Fill in `GITHUB_REPO_URL` and `HF_REPO_ID` in the next cell

In [1]:
# === CONFIG — change these two lines ===
GITHUB_REPO_URL = "https://github.com/nguyenvmthien/banking-intent-unsloth.git"
HF_REPO_ID      = "minhthien/banking-intent-unsloth"
# ========================================

import os
WORKING_DIR = f"/content/{GITHUB_REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')}"
print(f"Repo will be cloned to: {WORKING_DIR}")

Repo will be cloned to: /content/banking-intent-unsloth


## 1. Environment setup

In [2]:
import subprocess, sys

def run(cmd, capture=True):
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:] if capture else "")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout if capture else ""

run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q trl peft datasets langsmith huggingface_hub pandas scikit-learn tqdm pyyaml')
print("Dependencies installed.")

Dependencies installed.


In [3]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded.")

HF_TOKEN loaded.


In [4]:
from google.colab import drive
drive.mount("/content/drive")

# Checkpoints are saved here — survives session restarts
CHECKPOINT_DIR = "/content/drive/MyDrive/banking-intent-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/banking-intent-checkpoints


## 2. Clone repo and prepare data

In [5]:
run(f'git clone {GITHUB_REPO_URL} {WORKING_DIR}')
os.chdir(f"{WORKING_DIR}/scripts")
print(f"Repo cloned to {WORKING_DIR}")

Repo cloned to /content/banking-intent-unsloth


In [6]:
run('python preprocess_data.py')
print("Data ready.")

Data ready.


## 3. Configure HF repo for checkpoint push

In [7]:
import yaml

config_path = f"{WORKING_DIR}/configs/train.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID
config['output_dir'] = CHECKPOINT_DIR  # save to Drive instead of /content/

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id  : {HF_REPO_ID}")
print(f"output_dir    : {CHECKPOINT_DIR}")

hub_model_id  : minhthien/banking-intent-unsloth
output_dir    : /content/drive/MyDrive/banking-intent-checkpoints


## 4. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [8]:
import glob
from huggingface_hub import snapshot_download

output_dir = f"{WORKING_DIR}/outputs/checkpoint"
os.makedirs(output_dir, exist_ok=True)

has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, "checkpoint-*")))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")
else:
    latest = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))[-1]
    print(f"Local checkpoint found: {latest}")

No local checkpoint. Trying to restore from Hub: minhthien/banking-intent-unsloth


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Restored from Hub.


In [9]:
# os.chdir(f"{WORKING_DIR}/scripts")

# train_result = subprocess.run([sys.executable, "train.py"], env=os.environ)
# if train_result.returncode != 0:
#     raise RuntimeError("Training failed — check logs above.")
# print("Training complete.")

## 5. Quick sanity check — predict one sample

In [10]:
import sys
sys.path.insert(0, f"{WORKING_DIR}/scripts")

from inference import IntentClassification

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
test_input = "I lost my credit card, how do I order a replacement?"
result = clf.predict(test_input)
print(result)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
[FINETUNED] Loading model: ../outputs/checkpoint
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LangSmith tracing disabled.


Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

{'input': 'I lost my credit card, how do I order a replacement?', 'raw_output': 'card_about_to_expire', 'label': 'card_about_to_expire'}


## 6. Evaluate — quick test (200 samples)

Stratified sample across intents to verify the pipeline before running the full evaluation.

In [11]:
import sys, gc, yaml
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

sys.path.insert(0, f"{WORKING_DIR}/scripts")
from inference import IntentClassification

with open(f"{WORKING_DIR}/configs/inference.yaml") as f:
    config = yaml.safe_load(f)
batch_size = config.get("batch_size", 8)

test_df = pd.read_csv(f"{WORKING_DIR}/sample_data/test.csv")

# Stratified 200 samples: 3 per intent (77*3=231) then subsample to 200
sample_df = (
    test_df.groupby("intent_name", group_keys=False)
    .sample(n=3, random_state=42)
    .sample(n=200, random_state=42)
    .reset_index(drop=True)
)
print(f"Sampled {len(sample_df)} rows across {sample_df['intent_name'].nunique()} intents")


def run_eval(clf, df, label, print_per_sample=True):
    texts = df["text"].tolist()
    y_true = df["intent_name"].tolist()
    y_pred = []
    for start in tqdm(range(0, len(texts), batch_size), desc=label):
        results = clf.predict_batch(texts[start:start + batch_size])
        for result, true_label in zip(results, y_true[start:start + batch_size]):
            y_pred.append(result["label"])
            if print_per_sample:
                status = "OK" if result["label"] == true_label else "MISS"
                print(f"  [{status}] true={true_label}  pred={result['label']}", flush=True)
    acc = accuracy_score(y_true, y_pred)
    print(f"\n>>> {label} accuracy: {acc:.4f} ({int(acc*len(df))}/{len(df)})\n", flush=True)
    print(classification_report(y_true, y_pred, digits=4), flush=True)
    return acc


def free_model(clf):
    del clf.model
    del clf.tokenizer
    del clf
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed: {torch.cuda.memory_allocated()/1e9:.2f}GB used", flush=True)


print("=" * 60)
print("EVALUATE — 200 samples")
print("=" * 60)

results = {}

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="zero_shot")
results["zero_shot"] = run_eval(clf, sample_df, "zero_shot")
free_model(clf)

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
results["finetuned"] = run_eval(clf, sample_df, "finetuned")
free_model(clf)

print("=== SUMMARY ===")
for mode, acc in results.items():
    print(f"  {mode:<12} {acc:.4f}  ({acc*100:.2f}%)")


Sampled 200 rows across 77 intents
EVALUATE — 200 samples
[ZERO_SHOT] Loading model: unsloth/Qwen3-8B
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
LangSmith tracing disabled.


zero_shot:   0%|          | 0/25 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_

  [OK] true=virtual_card_not_working  pred=virtual_card_not_working
  [OK] true=change_pin  pred=change_pin
  [OK] true=apple_pay_or_google_pay  pred=apple_pay_or_google_pay
  [MISS] true=top_up_by_bank_transfer_charge  pred=transfer_fee_charged
  [OK] true=automatic_top_up  pred=automatic_top_up
  [OK] true=transfer_not_received_by_recipient  pred=transfer_not_received_by_recipient
  [MISS] true=beneficiary_not_allowed  pred=failed_transfer
  [OK] true=transfer_into_account  pred=transfer_into_account


...

  [MISS] true=wrong_amount_of_cash_received  pred=declined_cash_withdrawal
  [OK] true=virtual_card_not_working  pred=virtual_card_not_working
  [OK] true=top_up_by_bank_transfer_charge  pred=top_up_by_bank_transfer_charge
  [OK] true=card_swallowed  pred=card_swallowed
  [OK] true=card_payment_fee_charged  pred=card_payment_fee_charged
  [OK] true=direct_debit_payment_not_recognised  pred=direct_debit_payment_not_recognised
  [OK] true=balance_not_updated_after_cheque_or_cash_deposit  pred=balance_not_updated_after_cheque_or_cash_deposit
  [OK] true=card_swallowed  pred=card_swallowed


finetuned: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


>>> finetuned accuracy: 0.9000 (180/200)

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     1.0000    1.0000    1.0000         2
                                activate_my_card     1.0000    1.0000    1.0000         3
                                       age_limit     1.0000    1.0000    1.0000         3
                         apple_pay_or_google_pay     1.0000    1.0000    1.0000         3
                                     atm_support     1.0000    1.0000    1.0000         2
                                automatic_top_up     1.0000    0.6667    0.8000         3
         balance_not_updated_after_bank_transfer     0.0000    0.0000    0.0000         2
balance_not_updated_after_cheque_or_cash_deposit     1.0000    0.6667    0.8000         3
                         beneficiary_not_allowed     1.0000    0.6667    0.8000         3
                                 cancel_transfer     1.0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


VRAM freed: 0.11GB used
=== SUMMARY ===
  zero_shot    0.6800  (68.00%)
  finetuned    0.9000  (90.00%)


## 7. Evaluate on full test set

Run zero_shot and finetuned on all 3,080 samples.

In [15]:
print("\n" + "=" * 60)
print(f"FULL EVALUATION ({len(test_df)} samples)")
print("=" * 60)

full_results = {}
# for mode in ("zero_shot", "finetuned"):
# for mode in (f"finetuned"):
clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
full_results[mode] = run_eval(clf, test_df, mode, print_per_sample=False)

print("=== FULL EVALUATION SUMMARY ===")
for mode, acc in full_results.items():
    print(f"  {mode:<12} {acc:.4f}  ({acc*100:.2f}%)")



FULL EVALUATION (3080 samples)
[FINETUNED] Loading model: ../outputs/checkpoint
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
LangSmith tracing disabled.


f:   0%|          | 0/385 [00:00<?, ?it/s]Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE,


>>> f accuracy: 0.9185 (2829/3080)

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     0.9744    0.9500    0.9620        40
                                activate_my_card     1.0000    0.9250    0.9610        40
                                       age_limit     0.9524    1.0000    0.9756        40
                         apple_pay_or_google_pay     1.0000    1.0000    1.0000        40
                                     atm_support     0.9756    1.0000    0.9877        40
                                automatic_top_up     1.0000    0.9500    0.9744        40
         balance_not_updated_after_bank_transfer     0.7632    0.7250    0.7436        40
balance_not_updated_after_cheque_or_cash_deposit     1.0000    0.9250    0.9610        40
                         beneficiary_not_allowed     0.8611    0.7750    0.8158        40
                                 cancel_transfer     0.9750   